# Google Merchandise Store: a month in the life of a shop

Where do shoppers go after opening a product listing? What happens when they return to the basket to check out? We explore both questions with [retentioneering](https://retentioneering.com/docs) using the January 2021 extract of Google's [obfuscated ecommerce demo dataset](https://developers.google.com/analytics/bigquery/web-ecommerce-demo-dataset). It contains 493,416 events from 109,078 sessions and 92,983 user IDs. Google notes that obfuscation can limit consistency between fields.

**Reading the investigation**

| Section | Question |
|---|---|
| [1. Prepare the data](#gms-data) and [2. Build a funnel](#gms-funnel) | Which steps do sessions reach in order? |
| [3. Follow apparel landings](#gms-landings) | Why do so many paths end on the apparel hub? What can we test about that page? |
| [4. Follow sessions from the basket](#gms-checkout) | How much of the checkout drop can we locate before and after the shipping step? |

Two observations guide the investigation. About 88% of sessions that start on the apparel hub end after one step in the processed stream, compared with 56% for its sub-pages. After the first basket, 54% of sessions have neither a sign-in nor a shipping-details event. In each case we trace the counts back to the paths before considering a product change.

<a id="gms-data"></a>

## 1. Preparing and inspecting the data

Download the prepared January extract [`gms.csv.gz`](https://drive.google.com/file/d/19tAZNl6IROsqXaZlTLWX3WMSIUuDzh-J/view?usp=drive_link), put it beside this notebook, and run the cells in order. The examples use the Python package `retentioneering` and its Jupyter widgets.

The saved tables use Retentioneering 5.3.0. When events share a timestamp, their order in the processed stream matters for adjacent-event patterns.

One row represents a recorded event. A visitor can have several sessions, and a session can contain many visits to the same page. Whenever we report a session conversion, each eligible `session_id` counts once.

### 1.1 Loading the raw data

In [1]:
import pandas as pd
import retentioneering as rete

print("retentioneering", rete.__version__)

df = pd.read_csv("gms.csv.gz", compression="gzip", sep=";")
df["page_location"] = "https://shop.googlemerchandisestore.com" + df["page_location"]
df.head()

retentioneering 5.3.0


,user_id,session_id,event,timestamp,session_number,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location
0,1.000223e+09,1000223163.8035208_1,main,2021-01-07 18:35:34.089387,1,mobile,Australia,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
1,1.000223e+09,1000223163.8035208_1,main,2021-01-07 18:35:39.094280,1,mobile,Australia,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
2,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:04:59.887247,1,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
3,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:05:05.002377,1,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
4,1.000436e+07,10004358.089772267_1,PLP: Bags,2021-01-08 04:10:12.372423,1,mobile,Russia,google,cpc,<Other>,Bags,https://shop.googlemerchandisestore.com/Google...


### 1.2 Loading the data into an Eventstream

[Eventstream](https://retentioneering.com/docs/eventstream) is the central class of retentioneering.
It is essentially a thin wrapper around a `pandas.DataFrame` that knows what each column means. We declare that through the
[schema](https://retentioneering.com/docs/eventstream#schema):

- **`path_cols`** defines what counts as a single path. Usually these are columns such as `user_id` and `session_id`. Our data has both, so we put both into `path_cols`: `user_id` for behaviour over the whole period and `session_id` for a single visit. The first one becomes the default; every method takes `path_col="session_id"` to switch.
- **`segment_cols`** are categorical labels attached to events that split the eventstream into groups whose behaviour we may want to compare. In this dataset they are `country`, `utm_source`, `utm_medium`, `utm_campaign`, `product_type`.
- **`custom_cols`** are extra fields we will need later. Here it is `page_location`, which comes in very handy in case 1 (see section 3.2).

The `event` and `timestamp` columns already use the default names, so there is no need to mention them in the schema.

In [2]:
SCHEMA = {
    "path_cols": ["user_id", "session_id"],
    "segment_cols": [
        "device", "country", "utm_source", "utm_medium", "utm_campaign", "product_type",
    ],
    "custom_cols": ["page_location"],
}

stream0 = rete.Eventstream(df, schema=SCHEMA)
stream0.df.head()

,user_id,session_id,event,timestamp,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location,event_type,subindex,index
0,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:04:59.887247,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,raw,2.0,1
1,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:05:05.002377,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,raw,2.0,2
2,1.000557e+06,1000557.2911835024_1,main,2021-01-07 12:15:33.038847,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2.0,1
3,1.000557e+06,1000557.2911835024_1,view_promotion,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2.0,2
4,1.000557e+06,1000557.2911835024_1,main,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2.0,3


The underlying data is available through the `Eventstream.df` accessor. As you can see, three technical columns have been added to the original data: `event_type`, `index`, `subindex`.

The first method worth calling in any analysis is [`Eventstream.describe()`](https://retentioneering.com/docs/eventstream#describe). It reports the basics of the loaded eventstream: its size, date range, event frequencies, the distribution of path lengths and the segment levels.

In [3]:
overview = stream0.describe(top_events=12)

print(overview["shape"])
print(overview["date_range"])
display(overview["path_stats"]["user_id"])
display(overview["event_frequency"])

{'n_events': 493416, 'n_paths': 92983, 'n_unique_events': 47}
{'min': Timestamp('2021-01-01 00:00:08.136569'), 'max': Timestamp('2021-01-31 23:59:55.412363'), 'span': Timedelta('30 days 23:59:47.275794')}


,length,duration
count,92983.000000,9.298300e+04
mean,5.306518,3.612019e+04
std,10.437298,1.891713e+05
min,1.000000,0.000000e+00
25%,2.000000,4.894917e+00
50%,2.000000,5.045870e+00
75%,5.000000,6.255278e+01
90%,10.000000,3.211639e+03
99%,49.000000,1.129364e+06
max,593.000000,2.614472e+06


,event,count,share
0,main,116795,0.236707
1,PLP: Apparel,65391,0.132527
2,view_promotion,52209,0.105811
3,PDP: Apparel,36969,0.074925
4,PLP: Shop By Brand,25923,0.052538
5,store,23871,0.048379
6,PLP: Lifestyle,21323,0.043215
7,basket,18680,0.037859
8,add_to_cart,15179,0.030763
9,select_item,9949,0.020164


Two details will shape the analysis. First, the 47 event names include variants of `PLP` (product listing page) and `PDP` (product details page) for different categories. Second, half of the user paths have at most two events. Short paths will matter when we study where sessions end.

<a id="gms-funnel"></a>

## 2. An overview of user behaviour

### 2.1 A naive first step

Let us start by building one of the most important retentioneering widgets – the [transition graph](https://retentioneering.com/docs/widgets/transition-graph) – and see that applying it head-on gets us nowhere.

In [4]:
stream0.transition_graph(path_col="session_id")

TransitionGraphWidget — run the cell to open the widget.

Even with the [automatic edge filter](https://retentioneering.com/docs/widgets/transition-graph#edgefilter), 47 event names make the graph hard to follow.

### 2.2 Grouping product listings and product pages

A visit to `PLP: Bags` and a visit to `PLP: Drinkware` are both visits to a product listing. For the first pass, let us keep the level of the page and set aside its category. [`rename_events`](https://retentioneering.com/docs/data-processors/rename-events) groups 14 PLP names and 11 PDP names into two events, leaving 24 event names in the overview. The original stream keeps the finer detail for section 3.

In [5]:
plp_pdp = {e: e.split(":")[0] for e in stream0.df["event"].unique() if e.startswith(("PLP:", "PDP:"))}

list(plp_pdp.items())[:4]

[('PLP: Clearance', 'PLP'),
 ('PLP: Lifestyle', 'PLP'),
 ('PLP: Apparel', 'PLP'),
 ('PDP: Apparel', 'PDP')]

In [6]:
stream = stream0.rename_events(plp_pdp)
stream.describe()["shape"]

{'n_events': 493416, 'n_paths': 92983, 'n_unique_events': 24}

### 2.3 Collapsing repeated events within a session

A run of `shipping_details` events may reflect repeated edits to an address. A run of `add_to_cart` events may represent several products. At this stage we want to see what comes *after* each run. [`collapse_events(loops=True)`](https://retentioneering.com/docs/data-processors/collapse-events) represents consecutive identical event names as one step.

We pass `path_col="session_id"` so runs never cross session boundaries. From this point, “next step” means the next event in the processed session. At this level, even successive listings for different categories may collapse into one `PLP`.

In [7]:
stream = stream.collapse_events(loops=True, path_col="session_id")

print(f"{len(stream0.df):,} events -> {len(stream.df):,} after collapsing repeats")

493,416 events -> 325,330 after collapsing repeats


That made our eventstream about a third lighter.

> A filter we deliberately did *not* apply. The usual temptation is to drop very short
> paths, reasoning that "sessions of 1-2 events are noise and cannot be analysed anyway". In
> our case that would be a mistake. In case 1 we show that short sessions are exactly where one of the insights hides (see section 3).

### 2.4 The same graph, now readable

Here is what the transition graph looks like now:

In [8]:
stream.transition_graph(path_col="session_id")

TransitionGraphWidget — run the cell to open the widget.

Node positions on the graph are computed automatically, and they usually need a bit of manual work afterwards. To save the graph as a whole, use [`state_file`](https://retentioneering.com/docs/widgets#saving-widget-state). Everything about its state – manually arranged nodes, filters and so on – then survives a notebook restart:

```python
stream.transition_graph(state_file="gms_transition_graph.json", path_col="session_id")
```

Some structure of the CJM is visible now. To bring it out explicitly, let us highlight a path on the graph using the [views](https://retentioneering.com/docs/widgets/transition-graph#views) mechanism:

In [9]:
happy_path = ["PLP", "PDP", "add_to_cart", "basket", "shipping_details", "payment_details", "purchase"]

stream.transition_graph(
    views=[{"name": "Happy path", "focus": {"type": "path", "nodes": happy_path}}],
    view="Happy path",
    path_col="session_id"
)

TransitionGraphWidget — run the cell to open the widget.

The highlighted route connects adjacent events on the graph. A session could still visit each of these stages with searches, sign-in attempts or other events in between. Let us count those sessions with an ordered funnel.

### 2.5 Exploring the happy path

Let us now measure the conversion between the steps of the happy path with a [funnel](https://retentioneering.com/docs/widgets/funnel). Funnel steps are ordered and closed: a session reaches step N only if it has passed every previous step in order.

For convenience we split the path in two: before checkout and after.

In [10]:
stream.funnel(steps=["PLP", "PDP", "add_to_cart"], path_col="session_id", height=500)

FunnelWidget — run the cell to open the widget.

Of 109,078 sessions, 54,947 (50.4%) reach a listing, 8,503 (7.8%) have a product page after a listing, and 1,426 (1.3%) also have a subsequent `add_to_cart`. The last two percentages are shares of **all sessions** completing the indicated sequence. Among sessions that reach the preceding funnel step, the rates are 8,503 / 54,947 = 15.5% and 1,426 / 8,503 = 16.8%.

Now start a new funnel at the basket. A session can enter this one through any route.

In [11]:
stream.funnel(
    steps=["basket", "shipping_details", "payment_details", "purchase"],
    path_col="session_id",
    height=460,
)

FunnelWidget — run the cell to open the widget.

Of 6,743 basket sessions, 2,078 subsequently reach `shipping_details` (30.8%). From there 1,524 reach `payment_details` (73.3%), and 1,092 of those reach `purchase` (71.7%). The first step has the largest loss, in both session count and share. We return to it in section 4.

<a id="gms-landings"></a>

## 3. Why do we lose users while they are choosing a product?

### 3.1 Where sessions end

To see which steps precede `path_end`, the handy tools are [Step matrix](https://retentioneering.com/docs/widgets/step-matrix) and [Step Sankey](https://retentioneering.com/docs/widgets/step-sankey) with the `path_pattern="path_end"` argument, which aligns all paths on their last event.

In [12]:
stream.step_matrix(path_pattern="path_end", path_col="session_id", height=360)

StepMatrixWidget — run the cell to open the widget.

`PLP` is the last recorded event in 39% of sessions and `main` in 21%. The frequent `path_start` markers a few steps back also show that many sessions are short. Which listings account for the exits?

The transition graph can tell us much the same thing. The most convenient way to see which events precede `path_end` is [**Ego view**](https://retentioneering.com/docs/widgets/transition-graph#ego-view) focused on the `path_end` node. In this mode the graph unfolds the neighbourhood of a single event into a Sankey diagram and shows every incoming and outgoing edge, weighted by `proba_in` for incoming edges and `proba_out` for outgoing ones. With the focus on `path_end` only incoming edges show up, and they are exactly the events that end a session.

In [13]:
stream.transition_graph(
    path_col="session_id",
    views=[{
        "name": "Where sessions end",
        "egoNode": "path_end",                             # opens the Ego view dialog when the view is applied
        "focus": {"type": "node", "event": "path_end"},    # and keeps the node focused underneath it
    }],
    view="Where sessions end",
    height=560,
)

TransitionGraphWidget — run the cell to open the widget.

So we get the same top of session-ending events that we saw in the step matrix: `PLP` (39%), `main` (21%), `PDP` (16%).

### 3.2 Looking at individual pages

Let us dig into `PLP` and work out why this page so often turns out to be the last one in a session. Earlier we deliberately collapsed every category page into `PLP` to get a readable high-level picture. Now we need the opposite – to go down into specific categories and check whether some of those pages drive more churn than others. This is where the `page_location` column comes in, processed by the [`urls_to_events`](https://retentioneering.com/docs/data-processors/urls-to-events) data processor. It builds event names out of URLs and lets you collapse branches of the URL tree at any level.

Note the order of operations: first we split by URL, then we collapse repeated events, exactly as we did in the previous section.

In [14]:
all_pages = (
    stream0
    .urls_to_events("page_location", nodes=[], keep_full_paths=True)
    .collapse_events(loops=True, path_col="session_id")
)

print(f"{all_pages.df['event'].nunique()} distinct URL-labelled events")

931 distinct URL-labelled events


In [15]:
all_pages.df.head()

,user_id,session_id,event,timestamp,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location,event_type,index,subindex
0,1.000300e+06,1000299.7413851356_1,main://,2021-01-20 11:04:59.887247,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,collapsed,1,2.0
1,1.000557e+06,1000557.2911835024_1,main://,2021-01-07 12:15:33.038847,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,1,2.0
2,1.000557e+06,1000557.2911835024_1,view_promotion://,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2,2.0
3,1.000557e+06,1000557.2911835024_1,main://,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,3,2.0
4,1.000557e+06,1000557.2911835024_1,PLP: Clearance:/google+redesign/clearance,2021-01-07 12:15:56.604844,mobile,India,<Other>,<Other>,<Other>,Clearance,https://shop.googlemerchandisestore.com/Google...,raw,4,2.0


For each URL-labelled event, we will ask two questions. What share of *all session endings* comes from this event? And what share of *departures from this event* goes straight to `path_end`? The [transition graph data](https://retentioneering.com/docs/widgets/transition-graph#headless-mode) gives the first as `proba_in` and the second as `proba_out`. We add the number of recorded occurrences to put those percentages in context.

In [16]:
share_of_exits = all_pages.transition_graph_data(
    edge_weight="proba_in", path_col="session_id"
)["path_end"]
exit_rate = all_pages.transition_graph_data(
    edge_weight="proba_out", path_col="session_id"
)["path_end"]

exits = pd.DataFrame({
    "share of all exits": share_of_exits,
    "event occurrences": pd.Series(all_pages.get_event_counts()),
    "exit rate": exit_rate,
}).sort_values("share of all exits", ascending=False)

with pd.option_context("display.max_colwidth", 70):
    display(exits.head(10).round(3))

,share of all exits,event occurrences,exit rate
main://,0.213,72431.0,0.320
PLP: Apparel:/google+redesign/apparel,0.154,22986.0,0.733
view_promotion://,0.095,44429.0,0.232
PDP: Apparel:/google+redesign/apparel/google+dino+game+tee,0.065,7722.0,0.922
PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube,0.055,9458.0,0.630
store:/store.html,0.049,16469.0,0.326
PLP: Apparel:/google+redesign/apparel/mens,0.021,10719.0,0.212
PLP: Lifestyle:/google+redesign/lifestyle/drinkware,0.020,6193.0,0.359
basket:/basket.html,0.019,12328.0,0.171
PLP: Clearance:/google+redesign/clearance,0.018,8897.0,0.222


The apparel hub accounts for 15.4% of all session endings. Of the outgoing transitions from the hub, 73.3% go to `path_end`. For the men's listing the corresponding figures are 2.1% and 21.2%. Is the hub an unusually frequent exit because so many visits start there? Let us look at sessions that start on the hub and sessions that reach it later.

### 3.3 Exit rate split by scenario

To compare the hub with its own sub-pages, the two must first become two different events. The `aggregate_children` option of `urls_to_events` keeps the top-level URL `/google+redesign/apparel` as an event of its own and folds all of its sub-pages into `sub-page`. After that `rename_events` turns the URLs back into readable names.

In [17]:
pages = stream0.urls_to_events(
    "page_location",
    nodes=[{"path": "/google+redesign/apparel", "aggregate_children": True, "name": "sub-page"}],
).collapse_events(loops=True, path_col="session_id")

pages.df["event"].value_counts().head(8)

event
main://                                                      72431
view_promotion://                                            44429
PDP: Apparel:/google+redesign/apparel/sub-page               23449
PLP: Apparel:/google+redesign/apparel                        22986
PLP: Apparel:/google+redesign/apparel/sub-page               19336
store:/store.html                                            16469
basket:/basket.html                                          12328
PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube     9458
Name: count, dtype: int64

In [18]:
SPLIT = {
    "PLP: Apparel:/google+redesign/apparel": "Apparel hub",
    "PLP: Apparel:/google+redesign/apparel/sub-page": "Apparel sub-page",
    "PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube": "YouTube hub",
}

# keep the three splits above; for the rest, strip the URL and restore the plain name
pages = pages.rename_events({e: SPLIT.get(e, e.split(":")[0]) for e in pages.df["event"].unique()})

pages.df["event"].value_counts().head(8)

event
main                72515
PLP                 61900
view_promotion      50624
PDP                 41750
Apparel hub         22986
Apparel sub-page    19336
store               16487
basket              12439
Name: count, dtype: int64

With [`Eventstream.get_conversion_rate()`](https://retentioneering.com/docs/eventstream#get_conversion_rate) we can now measure the conversion from a given event (an anchor) `start_anchor` into `end_anchor="path_end"` happening on the very next step. The `start_anchor` values we care about are the following [patterns](https://retentioneering.com/docs/path-patterns):

- `path_start->PAGE` – the page was the first event of the session;
- `.->PAGE` – some real event preceded the page (`.` deliberately does not match `path_start`).

The `within=1` argument requires `end_anchor="path_end"` to happen exactly one step after the anchor set by `start_anchor`. A list of several `start_anchor` values measures the conversion into `end_anchor` for each of them.

Conversion here is always about paths: the share of paths in which `start_anchor` occurred and was then followed by `end_anchor`.

In [19]:
listing_pages = ["Apparel hub", "Apparel sub-page", "YouTube hub"]
page_anchors = [
    f"{lead}->{page}"
    for page in listing_pages
    for lead in ("path_start", ".")
]
next_step_exits = pages.get_conversion_rate(
    start_anchor=page_anchors, end_anchor="path_end", within=1, path_col="session_id",
)
display(next_step_exits[[
    "start_anchor", "paths_with_start", "converted", "conversion_rate"
]].round(3))

raw_session_sizes = df["session_id"].value_counts()
single_event_ids = raw_session_sizes.index[raw_session_sizes.eq(1)]
single_event_pages = pages.df.loc[pages.df["session_id"].isin(single_event_ids), "event"]
single_event_pages.value_counts().reindex(listing_pages, fill_value=0).rename("sessions with one raw event")

,start_anchor,paths_with_start,converted,conversion_rate
0,path_start->Apparel hub,17392,15292,0.879
1,.->Apparel hub,4340,1351,0.311
2,path_start->Apparel sub-page,2298,1277,0.556
3,.->Apparel sub-page,9835,2331,0.237
4,path_start->YouTube hub,7289,5200,0.713
5,.->YouTube hub,1834,668,0.364


event
Apparel hub         1271
Apparel sub-page     135
YouTube hub          189
Name: sessions with one raw event, dtype: int64

Of 17,392 sessions that start on the hub, 15,292 (87.9%) reach `path_end` on the very next processed step. For apparel sub-page landings that figure is 1,277 / 2,298 (55.6%). We grouped URL events and collapsed consecutive repeats. If a visitor views the hub repeatedly, several raw events can therefore appear as a single step. Only 1,271 hub landings and 135 sub-page landings contain exactly one raw event. Time spent on the page is a separate measurement.

When a hub visit follows another event, 1,351 / 4,340 sessions (31.1%) reach `path_end` on the step after the first match. A session can land on the hub and return to it later, so these two denominators can overlap. What happens if we follow the paths beyond that next step?

### 3.4 Funnel progress by landing page

To be thorough, let us also check whether the conversion from `path_start->Apparel hub` stays low not only in terms of ending the session quickly, but also in terms of moving down the funnel. This time we take `PDP`, `basket` and `purchase` as `end_anchor` instead of `path_end`, and we drop the `within=1` condition.

In [20]:
apparel_pages = ["Apparel hub", "Apparel sub-page"]
apparel_anchors = [
    f"{lead}->{page}"
    for lead in ("path_start", ".")
    for page in apparel_pages
]
landing_outcomes = pages.get_conversion_rate(
    start_anchor=apparel_anchors,
    end_anchor=["PDP", "basket", "purchase"],
    path_col="session_id",
)

landing_outcomes[[
    "start_anchor", "end_anchor", "paths_with_start", "converted", "conversion_rate", "lift"
]].round(4)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,lift
0,path_start->Apparel hub,PDP,17392,495,0.0285,0.1254
1,path_start->Apparel hub,basket,17392,230,0.0132,0.2139
2,path_start->Apparel hub,purchase,17392,22,0.0013,0.1262
3,path_start->Apparel sub-page,PDP,2298,446,0.1941,0.8548
4,path_start->Apparel sub-page,basket,2298,227,0.0988,1.5979
5,path_start->Apparel sub-page,purchase,2298,59,0.0257,2.5622
6,.->Apparel hub,PDP,4340,1281,0.2952,1.3000
7,.->Apparel hub,basket,4340,697,0.1606,2.5979
8,.->Apparel hub,purchase,4340,172,0.0396,3.9551
9,.->Apparel sub-page,PDP,9835,3743,0.3806,1.6763


Purchases follow 22 of 17,392 hub landings (0.13%) and 59 of 2,298 sub-page landings (2.57%). That is about a twentyfold difference. Product pages and baskets also appear more often after sub-page landings. Each outcome is measured separately from its starting anchor.

The `lift` column divides the observed conversion by the all-session prevalence of the same outcome. When we compare two groups for a single outcome, this common denominator cancels. The lift ratio and the conversion ratio therefore report the same contrast. Could the incoming audiences differ?

### 3.5 The landing page as a segment

The attribute "did this session start on `Apparel hub`" can be more than a one-off filter – we can record it as a [segment](https://retentioneering.com/docs/segments). The [`Eventstream.add_segment()`](https://retentioneering.com/docs/data-processors/add-segment) data processor with its `sql` argument assigns segment levels to rows however we like. Let us widen the attribute a little and define a segment with three levels, based on the first event of the session: `Apparel hub`, `Apparel sub-page` or `elsewhere` – they go into the `landing` column.

In [21]:
pages = pages.add_segment(
    "landing",
    sql="""
        SELECT CASE FIRST_VALUE(event) OVER (PARTITION BY session_id ORDER BY "index")
            WHEN 'Apparel hub' THEN 'Apparel hub'
            WHEN 'Apparel sub-page' THEN 'Apparel sub-page'
            ELSE 'elsewhere'
        END
        FROM eventstream
    """,
    path_col="session_id"
)

The [Segment overview](https://retentioneering.com/docs/widgets/segment-overview) compares session length, duration and event counts across landing groups. For the hub and sub-page *landing* rows in section 3.4, `has_event` reproduces the purchase, basket and product-page shares. The landing event identifies eligibility and each of these outcomes is a different event, so it can appear only later in that session. The rows anchored on later visits still need their own time origin.

Look also at the UTM rows. For each source, medium or campaign, the widget reports the share of sessions carrying that tag at least once. These shares give us a way to explore how incoming traffic is labelled.

In [22]:
pages.segment_overview(
    "landing",
    path_col="session_id",
    metrics=[
        {"metric": "length"},
        {"metric": "duration"},
        {"metric": "has_event", "metric_args": {"event": "PDP"}},
        {"metric": "has_event", "metric_args": {"event": "basket"}},
        {"metric": "has_event", "metric_args": {"event": "purchase"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_source"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_medium"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_campaign"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "device"}},
        {"metric": "in_segment", "metric_args": {"segment_name": "country", "segment_level": ["United States", "India", "Canada", "United Kingdom"]}},
    ],
)

SegmentOverviewWidget — run the cell to open the widget.

Each UTM row shows the share of sessions with the indicated tag at least once. A session may carry multiple tags. To see the source and medium together at the moment of entry, let us take the first event of each hub landing.

In [23]:
hub_sessions = (
    pages.df.loc[pages.df["landing"].eq("Apparel hub")]
    .sort_values(["session_id", "index", "subindex"])
    .drop_duplicates("session_id")
)
source_counts = hub_sessions.groupby(
    ["utm_source", "utm_medium"], observed=True, dropna=False
).size().sort_values(ascending=False)

source_counts.head(5).to_frame("sessions").assign(
    share_of_hub_landings=lambda table: table["sessions"] / len(hub_sessions)
).round(4)

sessions  share_of_hub_landings
utm_source                      utm_medium                                 
google                          organic         5968                 0.3431
(direct)                        (none)          4127                 0.2373
<Other>                         <Other>         2986                 0.1717
                                referral        1679                 0.0965
shop.googlemerchandisestore.com referral         929                 0.0534

`google / organic` is the largest recorded pair, accounting for 5,968 of 17,392 hub landings (34.3%). Two visitors with the same source and medium may have used different queries or come looking for different products. Let us compare the first steps they take in the shop.

### 3.6 Comparing the first steps

In a [step matrix in diff mode](https://retentioneering.com/docs/widgets#diff-mode), each cell shows the share in the first group minus the share in the second group for that event and step. We use hub landings as the first group and sub-page landings as the second.

In [24]:
landing_comparison = ("landing", "Apparel hub", "Apparel sub-page")
pages.step_matrix(
    diff=landing_comparison, path_col="session_id", step_window=5, height=380
)

StepMatrixWidget — run the cell to open the widget.

On the step after landing, `path_end` is more frequent for the hub by about 32 percentage points. To see the choices available to visitors who did continue, align their paths on the second step in the processed stream. [`truncate_paths`](https://retentioneering.com/docs/data-processors/truncate-paths) keeps just the sessions with such an event.

In [25]:
continued = pages.truncate_paths(
    start_anchor={"pattern": "path_start->.->.", "at": 2},
    end_anchor="path_end",
    path_col="session_id",
)
continued.step_matrix(
    diff=landing_comparison, path_col="session_id", step_window=5, height=380
)

StepMatrixWidget — run the cell to open the widget.

Among continuing sessions, the first aligned step after a hub landing shows more `search` and `view_search_results`, by 15.6 and 16.1 percentage points respectively. Apparel sub-pages also appear more often at that step. One hypothesis is that the showcase makes some visitors search for a category or item they expected to find directly. The search terms themselves would help us test that explanation.

For a view of the pages at the time, compare the [archived hub](https://web.archive.org/web/20210123165920/https://shop.googlemerchandisestore.com/Google+Redesign/Apparel) and [men's listing](https://web.archive.org/web/20210304130937/https://shop.googlemerchandisestore.com/Google+Redesign/Apparel/Mens).

### 3.7 What would we test on the apparel hub?

We followed the high share of paths ending on a listing to one particular page. The hub stands out as a landing page. Its landing sessions often have just one processed step, and among those that continue, search is a frequent next step. Audience mix, search intent and the design of the hub could all contribute to that pattern.

A product team could expose sub-categories and popular items on entry. Assign eligible visitors to the current or revised hub when they arrive, keep that assignment on repeat visits and compare product-page visits and purchases per assigned visitor over a fixed window. This would measure what the layout change does for people landing on the hub.

<a id="gms-checkout"></a>

## 4. From the basket to a purchase

What happens between opening the basket and completing an order? We will first follow a session from its first basket, then zoom in on individual returns to the basket, and finally put the paths back together. Every count in this section is a count of distinct sessions.

### 4.1 Where do paths diverge?

Start at the first `basket` in each session. We count the sessions in which each target event occurs later during the same visit. In every `get_conversion_rate` table below, `paths_with_start` gives the denominator for that row.

In [26]:
def percent_view(frame, *rate_columns):
    view = frame.copy()
    for name in rate_columns:
        view[name] = view[name].map("{:.2%}".format, na_action="ignore")
    return view

basket_baseline = stream.get_conversion_rate(
    start_anchor="basket", end_anchor=["shipping_details", "purchase"], path_col="session_id"
).set_index("end_anchor")
rate_columns = ["paths_with_start", "converted", "conversion_rate"]
percent_view(basket_baseline[rate_columns], "conversion_rate")

,paths_with_start,converted,conversion_rate
end_anchor,,,
shipping_details,6743,2078,30.82%
purchase,6743,1092,16.19%


Of 6,743 basket sessions, 2,078 (30.82%) subsequently reach the address step and 1,092 (16.19%) record a purchase. Someone might use the basket to keep an item for later, check the order total or start checkout. Let us see which next steps we can actually observe.

#### 4.1.1 Which event follows the basket?

Centre the step matrix on the first `basket` and read the column one step to its right.

In [27]:
stream.step_matrix(path_pattern="basket", path_col="session_id")

StepMatrixWidget — run the cell to open the widget.

`sign_in` and `path_end` each follow about 24% of first baskets. A product listing follows in 21%, a product page in 10% and `shipping_details` in 5%.

Now widen the question. A session may revisit the basket. Whenever the adjacent pair `basket → sign_in` or `basket → shipping_details` appears, how often does a purchase follow? Each row starts measurement at the end of its own first matching pair.

In [28]:
sign_in_pair = "basket->sign_in"
shipping_pair = "basket->shipping_details"
pair_rates = stream.get_conversion_rate(
    start_anchor=[sign_in_pair, shipping_pair], end_anchor="purchase", path_col="session_id"
).set_index("start_anchor")
percent_view(pair_rates[rate_columns], "conversion_rate")

,paths_with_start,converted,conversion_rate
start_anchor,,,
basket->sign_in,2258,681,30.16%
basket->shipping_details,1589,833,52.42%


The two rates are 681 / 2,258 (30.16%) and 833 / 1,589 (52.42%). The second group has already reached the address step. There is also a subtler point. A session can take one route, revisit the basket and take the other. Let us measure that overlap by keeping one flag per pattern for each session.

In [29]:
def session_flags(patterns):
    """Return a Boolean column per named pattern, indexed by session ID."""
    metrics = [
        {"metric": "matches_pattern", "metric_args": {"pattern": pattern}}
        for pattern in patterns.values()
    ]
    values = stream.get_metrics(metrics, path_col="session_id")
    columns = [f"matches_pattern_{pattern}" for pattern in patterns.values()]
    flags = values[columns].set_axis(list(patterns), axis="columns")
    assert flags.index.is_unique and flags.notna().all().all()
    return flags.astype(bool)

pairs = session_flags({"sign_in": sign_in_pair, "shipping": shipping_pair})
assert pairs.sum().tolist() == pair_rates.loc[
    [sign_in_pair, shipping_pair], "paths_with_start"
].tolist()
pd.Series({
    "sign_in only": (pairs["sign_in"] & ~pairs["shipping"]).sum(),
    "both": pairs.all(axis=1).sum(),
    "shipping only": (pairs["shipping"] & ~pairs["sign_in"]).sum(),
}, name="sessions")

sign_in only     1340
both              918
shipping only     671
Name: sessions, dtype: int64

#### 4.1.2 Which step accounts for the sign-in drop?

There are 918 sessions in both groups. Let us now stay with the 2,258 sessions containing `basket → sign_in` and ask two questions in sequence. How many subsequently reach shipping, and how many then purchase?

In [30]:
def checkout_stages(anchor):
    return session_flags({
        "eligible": anchor,
        "shipping": f"{anchor}->.*->shipping_details",
        "purchase": f"{anchor}->.*->purchase",
        "ordered_purchase": f"{anchor}->.*->shipping_details->.*->purchase",
    })

after_sign_in = checkout_stages(sign_in_pair)
after_sign_in.sum().rename("sessions")

eligible            2258
shipping            1397
purchase             681
ordered_purchase     681
Name: sessions, dtype: int64

In [31]:
assert after_sign_in["eligible"].equals(pairs["sign_in"])
assert after_sign_in["purchase"].sum() == pair_rates.at[sign_in_pair, "converted"]
bypass = after_sign_in["purchase"] & ~after_sign_in["ordered_purchase"]
assert not bypass.any(), f"{bypass.sum()} purchase sessions lack a shipping → purchase sequence"

Shipping follows sign-in in 1,397 / 2,258 sessions (61.87%). Of these, 681 / 1,397 (48.75%) go on to purchase. The ID check found an ordered shipping-then-purchase sequence in every session that purchased after this sign-in pair. That gives the factorisation

$$\frac{681}{2258} = \frac{1397}{2258}\times\frac{681}{1397}.$$

In 861 sessions shipping never followed the pair. Another 716 reached shipping and had no subsequent purchase. These are two distinct places to investigate.

Let us look at the next step for those 861 sessions. A pattern matching any sign-in pair with no later shipping would select 867 sessions. Six of them had already reached shipping after an earlier sign-in pair. Select the 861 by ID and centre the matrix on their first pair.

In [32]:
no_shipping = after_sign_in["eligible"] & ~after_sign_in["shipping"]
no_shipping_ids = after_sign_in.index[no_shipping].tolist()
stream.filter_events(keep={"session_id": no_shipping_ids}).step_matrix(
    anchor={"pattern": sign_in_pair, "at": -1}, path_col="session_id"
)

StepMatrixWidget — run the cell to open the widget.

After sign-in, about 31% of the 861 sessions end on the next step, 19% reach `registration` and 18% return to `basket`. Let us return to the full stream, where shipping is still possible, and compare subsequent outcomes for three next steps.

In [33]:
registration = f"{sign_in_pair}->registration"
next_steps = stream.get_conversion_rate(
    start_anchor=[registration, f"{sign_in_pair}->basket", f"{sign_in_pair}->PLP"],
    end_anchor=["shipping_details", "purchase"],
    path_col="session_id",
).set_index(["start_anchor", "end_anchor"])
percent_view(next_steps[rate_columns], "conversion_rate")

paths_with_start  converted  \
start_anchor                  end_anchor                                      
basket->sign_in->registration shipping_details               795        625   
                              purchase                       795        214   
basket->sign_in->basket       shipping_details               222         59   
                              purchase                       222         27   
basket->sign_in->PLP          shipping_details               102         14   
                              purchase                       102          4   

                                               conversion_rate  
start_anchor                  end_anchor                        
basket->sign_in->registration shipping_details          78.62%  
                              purchase                  26.92%  
basket->sign_in->basket       shipping_details          26.58%  
                              purchase                  12.16%  
basket->sign_in->PLP          shipping_details          13.73%  
                              purchase                   3.92%

Among the 795 sessions with `basket → sign_in → registration`, 625 subsequently reach shipping and 214 purchase. What happens between the registration event and returning to the order?

#### 4.1.3 How do sessions resume checkout after registration?

In [34]:
stream.step_matrix(path_pattern=registration, path_col="session_id", step_window=7)

StepMatrixWidget — run the cell to open the widget.

The matrix suggests a return via the storefront. We see `store` and `view_promotion` followed by another `basket`. Consider two ways of counting that return. The broad pattern accepts **zero or more** storefront events in the gap, so an immediate return also qualifies. The stricter pattern requires `store` somewhere in the gap. Both patterns count a session once, even if it makes several attempts.

In [35]:
storefront_gap = "[store|view_promotion]*"
basket_return = f"{registration}->{storefront_gap}->basket"
store_return = f"{registration}->{storefront_gap}->store->{storefront_gap}->basket"
routes = {
    "registration": checkout_stages(registration),
    "basket return": checkout_stages(basket_return),
    "basket return via store": checkout_stages(store_return),
}

assert (routes["basket return via store"]["eligible"] <= routes["basket return"]["eligible"]).all()
assert (routes["basket return"]["eligible"] <= routes["registration"]["eligible"]).all()
for route, flags in routes.items():
    assert (flags["ordered_purchase"] <= (flags["shipping"] & flags["purchase"])).all(), route
    assert flags["purchase"].equals(flags["ordered_purchase"]), f"Check purchase order for {route}"

route_counts = pd.DataFrame({name: flags.sum() for name, flags in routes.items()}).T
route_counts["purchase / eligible"] = route_counts["purchase"] / route_counts["eligible"]
route_counts["ordered_purchase / shipping"] = route_counts["ordered_purchase"] / route_counts["shipping"]
percent_view(route_counts, "purchase / eligible", "ordered_purchase / shipping")

,eligible,shipping,purchase,ordered_purchase,purchase / eligible,ordered_purchase / shipping
registration,795,625,214,214,26.92%,34.24%
basket return,549,499,180,180,32.79%,36.07%
basket return via store,514,478,169,169,32.88%,35.36%


In the broad return group, 499 of 549 sessions reach shipping after returning, and 180 of 549 (32.79%) purchase after returning. All 180 purchasing sessions contain at least one purchase after the shipping step. If we start the question at shipping, the rate is **180 / 499 = 36.07%**.

In 37 sessions the first broad match is a direct registration-to-basket return. Two of them also have a later return through `store`. Across all attempts, 514 sessions qualify for the stricter group. Of these, 478 reach shipping, and 169 purchase afterwards (35.36% of the 478). The log describes the route; establishing an automatic redirect would require a registration-success event and its navigation destination. To count all basket returns, allow any intervening event.

In [36]:
all_returns = stream.get_conversion_rate(
    start_anchor=registration, end_anchor="basket", path_col="session_id"
).set_index("end_anchor")
percent_view(all_returns[rate_columns], "conversion_rate")

,paths_with_start,converted,conversion_rate
end_anchor,,,
basket,795,710,89.31%


In total, 710 of 795 sessions revisit the basket by any route (89.31%). The other 85 have no recorded return in this session. A comparison conditioned on returning leaves these 85 out of its denominator. This is why the product test later in this section begins before navigation.

#### 4.1.4 How much does each branch contribute?

For the decomposition, return to the **first basket** in each session. Look for the next sign-in or shipping step. If sign-in comes first, look for the next registration or shipping step. These decisions define the six rows below. Other events can occur between decisions.

We will assign each basket session to exactly one row, then count purchases **after its first basket** in every row. This common outcome matters for paths whose branch pattern ends at `path_end`. Searching for a purchase after that boundary would produce a zero by construction.

In [37]:
first_basket = "path_start->[^basket]*->basket"
before_sign_or_shipping = "[^sign_in|shipping_details]*"
before_registration_or_shipping = "[^registration|shipping_details]*"
before_shipping = "[^shipping_details]*"

from_basket = f"{first_basket}->{before_sign_or_shipping}"
from_sign_in = f"{from_basket}->sign_in->{before_registration_or_shipping}"
from_registration = f"{from_sign_in}->registration->{before_shipping}"
branches = {
    "no sign-in or shipping": f"{from_basket}->path_end",
    "shipping before sign-in": f"{from_basket}->shipping_details",
    "sign-in → shipping": f"{from_sign_in}->shipping_details",
    "sign-in → registration → shipping": f"{from_registration}->shipping_details",
    "sign-in → registration, no shipping": f"{from_registration}->path_end",
    "sign-in, no registration or shipping": f"{from_sign_in}->path_end",
}

membership = session_flags(branches)
basket_ids = stream.df.loc[stream.df["event"].eq("basket"), "session_id"].unique()
expected_memberships = membership.index.isin(basket_ids).astype(int)
bad = membership.sum(axis=1).ne(expected_memberships)
assert not bad.any(), f"Incorrect branch membership for {membership.index[bad][:5].tolist()}"

`share` is the fraction of basket sessions in a branch. `contribution_pp` counts its purchasing sessions per 100 basket sessions. The contributions sum to the baseline purchase rate in percentage points. `magnitude_pp` compares each contribution with what that branch would contribute at the overall purchase rate; its sum is zero.

In [38]:
from math import isclose

outcomes = session_flags({
    "purchase": "basket->.*->purchase",
    "shipping": "basket->.*->shipping_details",
    "no_shipping": f"{first_basket}->{before_shipping}->path_end",
})
n = len(basket_ids)
p = int(outcomes["purchase"].sum())
assert n == basket_baseline.at["purchase", "paths_with_start"]
assert p == basket_baseline.at["purchase", "converted"]
shipping_branches = membership[["shipping before sign-in", "sign-in → shipping", "sign-in → registration → shipping"]]
assert shipping_branches.any(axis=1).equals(outcomes["shipping"])
remaining = membership.drop(columns=shipping_branches.columns)
assert remaining.any(axis=1).equals(outcomes["no_shipping"])

table = pd.DataFrame({
    "sessions": membership.sum(),
    "purchases": membership.loc[outcomes["purchase"]].sum(),
})
table["purchase_rate"] = table["purchases"] / table["sessions"]
table["share"] = table["sessions"] / n
table["contribution_pp"] = 100 * table["purchases"] / n
table["magnitude_pp"] = table["contribution_pp"] - table["share"] * (100 * p / n)

assert table["sessions"].sum() == n and table["purchases"].sum() == p
assert isclose(table["share"].sum(), 1.0)
assert isclose(table["contribution_pp"].sum(), 100 * p / n)
assert isclose(table["magnitude_pp"].sum(), 0.0, abs_tol=1e-10)
table.loc["total"] = [n, p, p / n, 1.0, 100 * p / n, 0.0]
table = table.astype({"sessions": int, "purchases": int})
view = percent_view(table, "purchase_rate", "share").round({"contribution_pp": 2, "magnitude_pp": 2})
view.set_axis([
    "1. No sign-in or shipping",
    "2. Shipping before sign-in",
    "3. Sign-in → shipping",
    "4. Sign-in → registration → shipping",
    "5. Sign-in → registration, no shipping",
    "6. Sign-in, no registration or shipping",
    "Total",
])

,sessions,purchases,purchase_rate,share,contribution_pp,magnitude_pp
1. No sign-in or shipping,3642,0,0.00%,54.01%,0.00,-8.75
2. Shipping before sign-in,630,377,59.84%,9.34%,5.59,4.08
3. Sign-in → shipping,807,492,60.97%,11.97%,7.30,5.36
4. Sign-in → registration → shipping,641,223,34.79%,9.51%,3.31,1.77
"5. Sign-in → registration, no shipping",216,0,0.00%,3.20%,0.00,-0.52
"6. Sign-in, no registration or shipping",807,0,0.00%,11.97%,0.00,-1.94
Total,6743,1092,16.19%,100.00%,16.19,0.00


The largest branch contains 3,642 / 6,743 sessions (54.0%) with neither sign-in nor shipping after the first basket. Another 1,023 (15.2%) record sign-in and have no shipping event anywhere after the first basket. A session that reached shipping first and visited sign-in later belongs to the shipping branch here. There are zero purchases after the first basket in these three branches.

Of the branches that reach shipping, the purchase rates are 377 / 630 (59.84%) when shipping comes before sign-in, 492 / 807 (60.97%) after sign-in without an intervening registration, and 223 / 641 (34.79%) after sign-in and registration. These rates describe sessions that made it to shipping. They give us a reason to investigate checkout access and resumption. The size of the effect of changing either experience requires a product test.

### 4.2 What happens in sessions that stop before shipping?

Let us revisit the 4,665 sessions with no `shipping_details` after the first basket. They form branches 1, 5 and 6 in the table above. Anchoring on the **first** basket matters. A session that reaches shipping, revisits the basket and then leaves has already reached our target.

We trim each path to the part from that first basket onwards. [Cluster analysis](https://retentioneering.com/docs/widgets/cluster-analysis) groups the resulting paths by their event counts. The counts tell us which events tend to occur together. To see their order, we can return to the individual paths.

In [39]:
post_basket_without_shipping = f"{first_basket}->{before_shipping}->path_end"
unfinished_checkout = stream.filter_paths(
    {
        "metric": "matches_pattern",
        "metric_args": {"pattern": post_basket_without_shipping},
        "op": "=",
        "value": True,
    },
    path_col="session_id",
).truncate_paths(start_anchor="basket", end_anchor="path_end", path_col="session_id")

expected_sessions = outcomes.index[outcomes["no_shipping"]]
assert set(unfinished_checkout.df["session_id"].unique()) == set(expected_sessions)

unfinished_checkout.cluster_analysis(
    features=[{"metric": "event_count_bulk"}],
    overview_metrics=[{"metric": "length"}, {"metric": "event_count_bulk"}],
    method="kmeans",
    method_args={"n_clusters": "2-8"},
    nmf_components="2-8",
    select={"n_clusters": 3, "nmf_components": 3},
    path_col="session_id",
)

ClusterAnalysisWidget — run the cell to open the widget.

In the widget we can explore different cluster counts. For the selected three-cluster solution, open the overview and compare the size, length and most frequent events in each group.

| Sessions | Share of the 4,665 | Mean path length from basket | What we see |
|---:|---:|---:|---|
| 4,271 | 91.6% | 3.24 | Short paths, sometimes with another basket visit, a listing or sign-in |
| 314 | 6.7% | 23.01 | Repeated product listings, product views and basket visits |
| 80 | 1.7% | 8.96 | Higher mean counts for shipping information, FAQ and returns pages |

For instance, `sign_in` averages 0.22 occurrences per session in the largest cluster. If we want the share of sessions with at least one sign-in, the corresponding metric is `has_event`.

The small group that visits shipping and returns pages suggests a question for a closer look. At which step do visitors open these pages, and what do they do next? Reading the paths or inspecting recorded errors could distinguish several explanations for the same counts.

### 4.3 Which product changes could we test?

Of the 6,743 basket sessions, 1,023 contain `sign_in` after the first basket and have no `shipping_details` anywhere after that first basket. The 549 sessions that returned to the basket along the registration pattern give us a different clue. Each observation points to a decision we could change in the product.

| Hypothesis | When to assign a visitor | Control and treatment |
|---|---|---|
| Offering guest checkout increases purchases. | At the first checkout prompt that requires an account, with an active basket. | Current sign-in prompt versus the same prompt with a guest checkout option. |
| Resuming checkout after successful registration increases purchases. | At confirmed registration success for a visitor registering from checkout with an active basket, before navigation. | Current destination versus resuming checkout with the same basket. |

For the second test, first record `registration_success`, what initiated the navigation, the intended and actual destinations, and whether the same basket was restored. Those events can establish how often an automatic redirect actually occurs.

For either test, randomise eligible visitors at the decision point and keep their assigned variant on repeat visits. Define the primary outcome as at least one purchase within a fixed window after assignment, for example 24 hours. Count every assigned visitor, including those who close the page or never return to the basket. Compare the purchase shares and report their difference in percentage points with a confidence interval. Reaching shipping and restoring the basket tell us where the journey changed. Monitor checkout errors and cancelled orders alongside purchases.

## 5. Conclusion

We started with a crowded transition graph and followed two routes worth investigating. About 88% of sessions that begin on the apparel hub contain a single step after the tutorial groups page events and collapses consecutive repeats. Comparing landings on the hub and its sub-pages gives the merchandising team a concrete question about which products or categories to show on arrival.

In checkout, repeated basket visits let the same session contribute to more than one transition pattern. Once we anchored the analysis on the first basket, the six branches accounted for every basket session, including all 1,092 sessions with a subsequent purchase. That decomposition locates the observed losses. The guest checkout and redirect tests give us a way to measure what changing the experience would achieve.